# TALSIM-NG Batch Simulation Runner

**Purpose:** Automates running all 143 KOSTRA precipitation scenarios through
TALSIM-NG in batch mode, reads the resulting `.MAX` and `.WEL` files, and
produces summary plots of peak outflow sorted by storm duration.

**What it does:**
- Reads programme IDs and scenario metadata from the `.PRO` file (KPRO block)
- Calls the TALSIM-NG simulation engine sequentially for each programme ID
- Reads `.MAX` files to extract peak outflow per scenario
- Groups results by return period (1yr – 100yr) and storm duration
- Plots peak discharge curves sorted by rainfall duration

**Dependencies:** `lib.timeseries`, `lib.talsim` (install via `pyproject.toml`)  
**Input:** TALSIM-NG project folder (`Ziegenrück_r`), TALSIM engine path  
**Output:** Peak outflow summary plots, console progress log

---

Batch simulate precipitation scenarios

In [ ]:
from pathlib import Path
import pandas as pd
from lib.timeseries import Timeseries
import lib.talsim as talsim
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator
import matplotlib.pyplot as plt
import numpy as np
import re
import matplotlib.cm as cm

In [ ]:
# settings
prog_ids = list(range(1,144))
dataset_dir = Path(r"C:\Users\raah\Desktop\Project_ZR\Ziegenrück_r")
dataset_name = "Ziegenrück_r"

engine_dir = Path(r"C:\Talsim-NG\Client\TalsimNG\SimEngine")

In [ ]:
from pathlib import Path

pro_file = Path(r"C:\Users\raah\Desktop\Project_ZR\Ziegenrück_r\Ziegenrück_r.PRO")
beschreibung_dict = {}

with open(pro_file, "r", encoding="latin-1") as f:
    lines = f.readlines()

# Find KPRO block
in_kpro = False
for line in lines:
    if line.strip() == "[KPRO]":
        in_kpro = True
        continue
    if line.startswith("[") and line.strip() != "[KPRO]":
        in_kpro = False
    if in_kpro and line.startswith(" |"):        # ← fixed: space + pipe
        id_field = line[4:7].strip()
        if not id_field.isdigit():               # ← skip header/comment rows
            continue
        prog_id = int(id_field)
        last_pipe = line.rfind("|")              # ← fixed: find last pipe
        beschreibung = line[last_pipe + 1:].strip()
        beschreibung_dict[prog_id] = beschreibung

# Simulation loop with folder names progID_Beschreibung
engine = talsim.TalsimEngine(engine_dir)
tds = talsim.TalsimDataset(dataset_dir, dataset_name)

for prog_id in prog_ids:
    print(f"Simulating ProgId {prog_id}...")
    tds.set_sim_options({"KProgID": prog_id})
    engine.simulate(tds)
    beschreibung = beschreibung_dict.get(prog_id, "NoDesc")
    dir_results = tds.path / f"{prog_id:0>3}_{beschreibung}"
    tds.copy_result_files(dir_results)

In [ ]:
result_var = "S004_1AB"

#selected_progids = [5, 18, 57, 83, 96, 122, 31, 44, 70, 109, 135]  # 1yr
#selected_progids = [9, 22, 61, 87, 100, 126, 35, 48, 74, 113, 139]  # 2yr
#selected_progids = [13, 26, 65, 91, 104, 130, 39, 52, 78, 117, 143]  # 5yr
#selected_progids = [4, 17, 56, 82, 95, 121, 30, 43, 69, 108, 134]  # 10yr
#selected_progids = [8, 21, 60, 86, 99, 125, 34, 47, 73, 112, 138]  # 20yr
#selected_progids = [12, 25, 64, 90, 103, 129, 38, 51, 77, 116, 142]  # 50yr
#selected_progids = [3, 16, 55, 81, 94, 120, 29, 42, 68, 107, 133]  # 100yr
#selected_progids = [7, 20, 59, 85, 98, 124, 33, 46, 72, 111, 137]  # 200yr
#selected_progids = [11, 24, 63, 89, 102, 128, 37, 50, 76, 115, 141]  # 500yr
#selected_progids = [2, 15, 54, 80, 93, 119, 28, 41, 67, 106, 132]  # 1000yr
#selected_progids = [6, 19, 58, 84, 97, 123, 32, 45, 71, 110, 136]  # 2000yr
selected_progids = [ 10,23,62, 88, 101, 127, 36, 49, 75, 114, 140]  # 5000yr 
#selected_progids = [1, 14, 53, 79, 92, 118, 27, 40, 66, 105, 131]  # 10000yr

ts_list = []
legends = []

# -------------------------------
# Read WEL files and prepare legend names
# -------------------------------
for prog_id in selected_progids:
    beschreibung = beschreibung_dict.get(prog_id, "NoDesc")
    dir_results = tds.path / f"{prog_id:0>3}_{beschreibung}"
    file_wel = dir_results / f"{tds.name}.WEL"
    
    if not file_wel.exists():
        print(f"File not found: {file_wel}")
        continue
    
    ts, = Timeseries.read_wel(file_wel, [result_var])
    ts.title += f" ({prog_id:0>3}_{beschreibung})"
    ts_list.append(ts)
    
    # Simplify legend: take first part before '_' and add space before unit
    legend_name = beschreibung.split("_")[0]
    for unit in ["h", "d", "min"]:  # handle common units
        if unit in legend_name:
            legend_name = legend_name.replace(unit, f" {unit}")
    legends.append(legend_name)

# -------------------------------
# Convert timeseries list to DataFrame
# -------------------------------
df_results = Timeseries.to_dataframe(ts_list)

# Convert index to hours
time_hours = (df_results.index - df_results.index[0]).total_seconds() / 3600
df_results_hours = df_results.copy()
df_results_hours.index = time_hours
df_results_hours.index.name = "Hour"


# Extract numeric return period for sorting
def get_dauer_numeric(legend_name):
    m = re.match(r"([0-9.]+)", legend_name)
    return float(m.group(1)) if m else float('inf')

# Sort columns and legends by return period
sorted_pairs = sorted(
    zip(df_results_hours.columns, legends),
    key=lambda x: get_dauer_numeric(x[1])
)
sorted_columns, sorted_legends = zip(*sorted_pairs)

# -------------------------------
# Plotting
# -------------------------------
fig, ax = plt.subplots(figsize=(14, 7))

for col, legend_name in zip(sorted_columns, sorted_legends):
    # Format legend as Dauerstufe
    legend_label = f"Dauerstufe {legend_name}"
    ax.plot(
        df_results_hours.index,
        df_results_hours[col],
        linewidth=2,
        label=legend_label
    )

ax.set_xlabel("Zeit [h]", fontsize=14)
ax.set_ylabel("Q [m³/s]", fontsize=14)
#ax.set_ylabel("[Tsd m³]", fontsize=14)
ax.set_title(f"TALSIM-NG: Zufluss TS_Neuer_Teich, Jährlichkeit 1000A", fontsize=16, pad=15)
#ax.set_title(f"TS Neuer_Teich, Jährlichkeit 10000A", fontsize=16, pad=15)

ax.grid(True, which="major", color="0.7", linestyle="-", linewidth=1)
ax.tick_params(axis="both", labelsize=12)

ax.legend(
    loc="upper right",
    bbox_to_anchor=(1.0, 1.0),
    frameon=False,
    ncol=2,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MAX FILE READER – OUTFLOW1 (Qab_1) AT S020
# SORTED BY RAINFALL DURATION (h)
# ============================================================

import pandas as pd
import re


def read_max_outflow(file_max, element_key="T001"):
    """
    Reads a TALSIM .MAX file and returns Outflow1 (Qab_1)
    for a given element key (default: S020).
    """

    # --- find header line dynamically ---
    with open(file_max, "r", encoding="latin-1") as f:
        lines = f.readlines()

    header_line_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("No;"):
            header_line_idx = i
            break

    if header_line_idx is None:
        raise ValueError("Could not find MAX table header")

    # --- read table ---
    df = pd.read_csv(
        file_max,
        sep=";",
        skiprows=header_line_idx,
        engine="python"
    )

    # Clean column names
    df.columns = [c.strip() for c in df.columns]

    # Remove unit/empty rows
    df = df[df["No"].astype(str).str.strip().str.isnumeric()]

    # Clean Key column
    df["Key"] = df["Key"].astype(str).str.strip()

    # Select required element
    row = df[df["Key"] == element_key]

    if row.empty:
        raise ValueError(f"Element '{element_key}' not found in MAX file")

    return float(row["Inflow"].values[0])


# -----------------------------
# READ ALL MAX FILES
# -----------------------------

max_results = []

for prog_id in selected_progids:
    beschreibung = beschreibung_dict.get(prog_id, "NoDesc")
    dir_results = tds.path / f"{prog_id:03d}_{beschreibung}"
    file_max = dir_results / f"{tds.name}.MAX"

    if not file_max.exists():
        continue

    qmax = read_max_outflow(file_max, element_key="T001")

    # Extract duration in hours from description (e.g. 0.25h, 12h, 72h)
    match = re.search(r"([\d\.]+)h", beschreibung)
    duration_h = float(match.group(1)) if match else None

    max_results.append({
        "ProgID": prog_id,
        "Beschreibung": beschreibung,
        "Duration_h": duration_h,
        "Qmax_T001_m3s": qmax
    })


# -----------------------------
# SORT BY DURATION
# -----------------------------

max_results_sorted = sorted(
    max_results,
    key=lambda x: x["Duration_h"]
)

df_qmax = pd.DataFrame(max_results_sorted)


# -----------------------------
# PRINT RESULTS (ORDERED)
# -----------------------------


print("\nResult DataFrame:")
display(df_qmax)